In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
project = os.getenv('GOOGLE_CLOUD_PROJECT')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",
                              temperature=0.5,
vertexai=True,
project=project
)

In [4]:
from langchain.tools import tool
@tool
def say_hello(name: str) -> str:
    """Returns a greeting message for the given name."""
    return f"Hello, {name}!"

In [6]:
llm_with_tools = llm.bind_tools([say_hello])

In [7]:
response =llm.invoke("Greet Shyam")
response


AIMessage(content='Here are a few ways to greet Shyam, ranging from casual to a bit more formal:\n\n**Casual:**\n\n*   "Hey Shyam!"\n*   "Hi Shyam!"\n*   "What\'s up, Shyam?"\n*   "Hello Shyam!"\n\n**Slightly More Formal/Polite:**\n\n*   "Good morning/afternoon/evening, Shyam." (Use the appropriate time of day)\n*   "Hello Shyam, how are you?"\n*   "Hi Shyam, hope you\'re doing well."\n\n**If you know him well and want to be friendly:**\n\n*   "Shyam! Good to see you!"\n*   "Hey there, Shyam!"\n\nChoose the one that best fits your relationship with Shyam and the context of the greeting!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d6283-5936-7440-917a-ea586d410037-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 164, 'total_tokens': 166, 'input_token_details': {'cache_read': 0}})

In [10]:
response = llm_with_tools.invoke("Greet Shyam")
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "Shyam"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d6284-1e0e-7be3-bf0c-4e06993018d6-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'Shyam'}, 'id': '09c80315-a370-4a71-b847-1ef64c4b9615', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 6, 'total_tokens': 24, 'input_token_details': {'cache_read': 0}})

In [11]:
from langchain.agents import create_agent
agent = create_agent(
    model = llm,
    tools = [say_hello]
)

In [12]:
agent_response = agent.invoke({
   "messages": [
       ("user", "Greet Shyam")
    ]

})



In [13]:
agent_response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Hello, Shyam!


In [21]:
load_dotenv()
project = os.getenv('TAVILY_API_KEY')

In [22]:
from langchain_tavily import TavilySearch
tavily_search_tool = TavilySearch(
    max_results = 3,
    topic = "news"
)

In [23]:
agent = create_agent(
    model=llm,
    tools=[tavily_search_tool]
)

In [24]:
response = agent.invoke({
    "messages": "Get me latest news about gpu's "
})
response['messages']

[HumanMessage(content="Get me latest news about gpu's ", additional_kwargs={}, response_metadata={}, id='333faaad-d86d-4467-8b2e-1419e98fbdd3'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "gpu news", "topic": "news", "time_range": "day"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d62a7-d2cd-7d81-a2a5-466c1e169a26-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'gpu news', 'topic': 'news', 'time_range': 'day'}, 'id': '71c6b82a-8744-4e36-b2a2-2f2102789db1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1298, 'output_tokens': 14, 'total_tokens': 1312, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"query": "gpu news", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.businessinsider.com/ai-demand-boosts-gp

In [25]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here are the latest news about GPUs:

*   GPU prices have seen a significant increase due to high demand for AI, impacting both older and newer Nvidia GPUs. This trend is driven by the substantial investments tech giants and AI labs are making in GPUs and data centers to support the generative AI boom.
*   Nvidia is developing new AI texture and material systems that aim to reduce asset size, speed up shading, and conserve GPU resources. These systems use neural representations to decrease memory usage and improve performance, potentially offering substantial advancements for game development.
*   10 Federal has appointed a top-ranked Nvidia AI engineer as its Chief AI Officer, marking a significant move for the self-storage industry.


In [26]:
@tool
def find_friends(name:str) -> list:
    """Find friends of a person 
    """
    return ["Amar", "Akbar", "Anthony"]

In [27]:
agent = create_agent(
    model=llm,
    tools = [find_friends]
)

In [28]:
response = agent.invoke({
    "messages": "Find friends of Shyam"
})

In [29]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here are Shyam's friends: Amar, Akbar, and Anthony.
